In [0]:
dbutils.widgets.text("catalog", "anurag_dev")
catalog = dbutils.widgets.get("catalog")

# OPTIMIZE with Z-ORDER (Task 5)
spark.sql(f"OPTIMIZE {catalog}.silver.orders_enriched ZORDER BY (state, order_date)")
spark.sql(f"OPTIMIZE {catalog}.gold.fact_sales ZORDER BY (category, order_date)")

# VACUUM (remove old files, default 7 days)
spark.sql(f"VACUUM {catalog}.silver.orders_enriched")
spark.sql(f"VACUUM {catalog}.gold.fact_sales")

# TIME TRAVEL validation
spark.sql(f"DESCRIBE HISTORY {catalog}.gold.fact_sales").show(5)
df_v0 = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .table(f"{catalog}.gold.fact_sales")
print(f"Version 0 row count: {df_v0.count()}")

# SCHEMA ENFORCEMENT (auto-enabled in Delta — verify)
spark.sql(f"DESCRIBE DETAIL {catalog}.gold.fact_sales").select("format","numFiles","sizeInBytes").show()